Author: Rachna

Project: CECE-FMT

# FastSpar Correlation Pipeline (Colab)

This notebook performs microbial co-occurrence analysis using FastSpar.

### Steps:
- Install FastSpar
- Prepare species-level count tables
- Run SparCC correlations
- Generate bootstrap samples
- Compute p-values

### Input:
- Species-level count matrices (CD, UC, Healthy)

### Output:
- Correlation matrices (`cor_*.csv`)
- P-value matrices (`pvals_*.csv`)
- Bootstrap results

### Notes:
- Data must be in BIOM-compatible TSV format
- Species-level filtering is applied

In [1]:
!apt-get update -y -qq
!apt-get install -y -qq libgsl-dev build-essential

!git clone https://github.com/scwatts/fastspar.git

%cd fastspar
!./autogen.sh
!./configure
!make -j4

!./src/fastspar --version

print("FastSpar ready")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package libgslcblas0:amd64.
(Reading database ... 118194 files and directories currently installed.)
Preparing to unpack .../libgslcblas0_2.7.1+dfsg-3_amd64.deb ...
Unpacking libgslcblas0:amd64 (2.7.1+dfsg-3) ...
Selecting previously unselected package libgsl27:amd64.
Preparing to unpack .../libgsl27_2.7.1+dfsg-3_amd64.deb ...
Unpacking libgsl27:amd64 (2.7.1+dfsg-3) ...
Selecting previously unselected package libgsl-dev.
Preparing to unpack .../libgsl-dev_2.7.1+dfsg-3_amd64.deb ...
Unpacking libgsl-dev (2.7.1+dfsg-3) ...
Setting up libgslcblas0:amd64 (2.7.1+dfsg-3) ...
Setting up libgsl27:amd64 (2.7.1+dfsg-3) ...
Setting up libgsl-dev (2.7.1+dfsg-3) ...
Processing triggers for man-db (2.10.2-1) ...
Processing triggers for libc-bin (2.35-0ubuntu3.13) ...
/sbin/ldconfig.

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import os

BASE = "/content/drive/MyDrive"

cd_file = f"{BASE}/CD_counts.tsv"
uc_file = f"{BASE}/UC_counts.tsv"
h_file  = f"{BASE}/Healthy_counts.tsv"

print("Files ready:")
print(cd_file, os.path.exists(cd_file))
print(uc_file, os.path.exists(uc_file))
print(h_file, os.path.exists(h_file))

Files ready:
/content/drive/MyDrive/CD_counts.tsv True
/content/drive/MyDrive/UC_counts.tsv True
/content/drive/MyDrive/Healthy_counts.tsv True


In [5]:
import os
import shutil
import pandas as pd

print("Copying count files from Drive → Colab\n")

for cond in ["CD", "UC", "Healthy"]:
    src = f"/content/drive/MyDrive/{cond}_counts.tsv"
    dst = f"/content/{cond}_counts.tsv"

    if os.path.exists(src):
        shutil.copy(src, dst)

        df = pd.read_csv(dst, sep="\t", index_col=0)

        print(f"{cond}: copied successfully")
        print(f"   Shape: {df.shape}")
    else:
        print(f"{cond}: NOT FOUND")
        print(f"   Expected at: {src}")

Copying count files from Drive → Colab

CD: copied successfully
   Shape: (748, 94)
UC: copied successfully
   Shape: (456, 93)
Healthy: copied successfully
   Shape: (428, 101)


In [9]:
import pandas as pd

for cond in ["CD", "UC", "Healthy"]:
    path = f"/content/{cond}_counts.tsv"

    df = pd.read_csv(path, sep="\t", index_col=0)

    df = df.T

    df.reset_index(inplace=True)

    df.rename(columns={"index": "#OTU ID"}, inplace=True)

    df.to_csv(path, sep="\t", index=False)

    print(f"Fixed format for {cond}: {df.shape}")

Fixed format for CD: (94, 749)
Fixed format for UC: (93, 457)
Fixed format for Healthy: (101, 429)


In [10]:
pd.read_csv("/content/CD_counts.tsv", sep="\t").head()

,#OTU ID,CSM5MCVL,CSM5MCVN,CSM5MCW6,CSM5MCWC,CSM5MCWE,CSM5MCWG,CSM5MCXD,CSM5MCXT,CSM5MCXV,...,CSM5MCU4_P,MSM5LLHI_P,ESM5ME9D_P,MSM5LLHO_P,CSM5MCVJ_P,CSM5MCWI_P,MSM5LLHA_P,HSM5MD4A_P,ESM5MEBA_P,HSM5FZC2_P
0,Akkermansia_muciniphila,0,0,0,0,0,0,323,0,0,...,0,353,0,176,0,22,17,612,0,0
1,Alistipes_finegoldii,0,0,0,0,0,0,0,2,5,...,0,1120,0,630,0,0,13,0,0,0
2,Alistipes_indistinctus,0,0,0,0,0,0,0,0,0,...,0,16,0,7,0,0,0,0,0,0
3,Alistipes_onderdonkii,0,0,0,0,0,0,0,9,28,...,0,115,0,111,0,0,4,0,0,80
4,Alistipes_putredinis,0,0,0,1,0,0,0,811,877,...,0,643,0,1103,0,0,1,0,0,527


In [11]:
import os, subprocess, glob, time

def run_full_fastspar(condition, n_bootstrap=50, iterations=20):

    counts = f"/content/{condition}_counts.tsv"

    base_out = f"/content/fastspar/sparcc_{condition}"
    boot_counts = f"{base_out}/bootstrap_counts"
    boot_cors = f"{base_out}/bootstrap_cors"
    boot_only = f"{base_out}/bootstrap_corr_only"

    os.makedirs(base_out, exist_ok=True)
    os.makedirs(boot_counts, exist_ok=True)
    os.makedirs(boot_cors, exist_ok=True)
    os.makedirs(boot_only, exist_ok=True)

    print(f"\n{'='*50}")
    print(f"Processing: {condition}")
    print('='*50)

    print("Step 1/4: Main correlations...")

    r = subprocess.run([
        "./src/fastspar",
        "--otu_table", counts,
        "--correlation", f"{base_out}/cor_{condition}.csv",
        "--covariance", f"{base_out}/cov_{condition}.csv",
        "--iterations", str(iterations),
        "--yes"
    ], cwd="/content/fastspar", capture_output=True, text=True)

    if r.returncode != 0:
        print("ERROR in main correlation:")
        print(r.stderr[:300])
        return False

    print("Done")

    print(f"Step 2/4: Generating {n_bootstrap} bootstraps...")

    subprocess.run("rm -f /content/fastspar/boot_*.tsv", shell=True)

    r = subprocess.run([
        "./src/fastspar_bootstrap",
        "--otu_table", counts,
        "--number", str(n_bootstrap),
        "--prefix", "boot_"
    ], cwd="/content/fastspar", capture_output=True, text=True)

    if r.returncode != 0:
        print("ERROR in bootstrap generation:")
        print(r.stderr[:300])
        return False

    boot_files = glob.glob("/content/fastspar/boot_*.tsv")

    if len(boot_files) == 0:
        print("CRITICAL: No bootstrap files generated!")
        return False

    for f in boot_files:
        subprocess.run(["mv", f, boot_counts])

    print(f"{len(boot_files)} bootstrap files created")

    boot_files = sorted(glob.glob(f"{boot_counts}/boot_*.tsv"))

    print(f"Step 3/4: Running bootstrap correlations ({len(boot_files)})...")

    for i, bf in enumerate(boot_files):
        base = os.path.basename(bf).replace(".tsv", "")

        subprocess.run([
            "./src/fastspar",
            "--otu_table", bf,
            "--correlation", f"{boot_cors}/{base}.tsv",
            "--covariance", f"{boot_cors}/{base}_cov.tsv",
            "--iterations", str(iterations),
            "--yes"
        ], cwd="/content/fastspar", capture_output=True)

        if (i + 1) % 10 == 0:
            print(f"  Progress: {i+1}/{len(boot_files)}")

    print("Bootstrap correlations done")


    for f in glob.glob(f"{boot_cors}/*.tsv"):
        if "_cov" not in f:
            subprocess.run(["cp", f, boot_only])

    print(f"{len(os.listdir(boot_only))} correlation files ready")

    print("Step 4/4: Computing p-values...")

    r = subprocess.run([
        "./src/fastspar_pvalues",
        "--otu_table", counts,
        "--correlation", f"sparcc_{condition}/cor_{condition}.csv",
        "--prefix", f"sparcc_{condition}/bootstrap_corr_only/boot_",
        "--permutations", str(n_bootstrap),
        "--outfile", f"sparcc_{condition}/pvals_{condition}.csv"
    ], cwd="/content/fastspar", capture_output=True, text=True)

    if r.returncode != 0:
        print("ERROR in p-values:")
        print(r.stderr[:300])
        return False

    print("P-values computed successfully")

    return True


start = time.time()

for cond in ["CD", "UC", "Healthy"]:
    success = run_full_fastspar(cond, n_bootstrap=50, iterations=20)

    if success:
        print(f"{cond} COMPLETE")
    else:
        print(f"{cond} FAILED")

print(f"\nTotal time: {(time.time() - start)/60:.1f} minutes")


Processing: CD
Step 1/4: Main correlations...
Done
Step 2/4: Generating 50 bootstraps...
50 bootstrap files created
Step 3/4: Running bootstrap correlations (50)...
  Progress: 10/50
  Progress: 20/50
  Progress: 30/50
  Progress: 40/50
  Progress: 50/50
Bootstrap correlations done
50 correlation files ready
Step 4/4: Computing p-values...
P-values computed successfully
CD COMPLETE

Processing: UC
Step 1/4: Main correlations...
Done
Step 2/4: Generating 50 bootstraps...
50 bootstrap files created
Step 3/4: Running bootstrap correlations (50)...
  Progress: 10/50
  Progress: 20/50
  Progress: 30/50
  Progress: 40/50
  Progress: 50/50
Bootstrap correlations done
50 correlation files ready
Step 4/4: Computing p-values...
P-values computed successfully
UC COMPLETE

Processing: Healthy
Step 1/4: Main correlations...
Done
Step 2/4: Generating 50 bootstraps...
50 bootstrap files created
Step 3/4: Running bootstrap correlations (50)...
  Progress: 10/50
  Progress: 20/50
  Progress: 30/50
  P

In [12]:
import os

for cond in ["CD", "UC", "Healthy"]:
    path = f"/content/fastspar/sparcc_{cond}/bootstrap_counts"
    print(f"\n{cond}:")
    if os.path.exists(path):
        files = os.listdir(path)
        print(f"Files found: {len(files)}")
        print(files[:5])
    else:
        print("Folder not found")


CD:
Files found: 50
['boot__7.tsv', 'boot__12.tsv', 'boot__17.tsv', 'boot__2.tsv', 'boot__8.tsv']

UC:
Files found: 50
['boot__7.tsv', 'boot__12.tsv', 'boot__17.tsv', 'boot__2.tsv', 'boot__8.tsv']

Healthy:
Files found: 50
['boot__7.tsv', 'boot__12.tsv', 'boot__17.tsv', 'boot__2.tsv', 'boot__8.tsv']


In [13]:
import os

for cond in ["CD", "UC", "Healthy"]:
    path = f"/content/fastspar/sparcc_{cond}"
    print(f"\n{cond}:")
    print(os.listdir(path))


CD:
['bootstrap_cors', 'bootstrap_corr_only', 'cor_CD.csv', 'bootstrap_counts', 'cov_CD.csv', 'pvals_CD.csv']

UC:
['bootstrap_cors', 'bootstrap_corr_only', 'pvals_UC.csv', 'bootstrap_counts', 'cov_UC.csv', 'cor_UC.csv']

Healthy:
['bootstrap_cors', 'bootstrap_corr_only', 'pvals_Healthy.csv', 'cor_Healthy.csv', 'bootstrap_counts', 'cov_Healthy.csv']


In [20]:
print("VERIFICATION:")

for cond in ["CD", "UC", "Healthy"]:
    cor_path = f"/content/fastspar/sparcc_{cond}/cor_{cond}.csv"
    pval_path = f"/content/fastspar/sparcc_{cond}/pvals_{cond}.csv"

    if os.path.exists(cor_path) and os.path.exists(pval_path):
        cor = pd.read_csv(cor_path, sep=None, engine='python', index_col=0)
        pval = pd.read_csv(pval_path, sep=None, engine='python', index_col=0)

        pval = pval.apply(pd.to_numeric, errors='coerce').fillna(1.0)

        print(f"\nDEBUG {cond}: shape = {cor.shape}")

        sig = ((cor.abs() > 0.3) & (pval < 0.05))
        n_sig = sig.values.sum() // 2

        print(f"{cond}:")
        print(f"  Matrix shape: {cor.shape}")
        print(f"  Significant edges: {n_sig}")

    else:
        print(f"✗ {cond}: files missing")

VERIFICATION:

DEBUG CD: shape = (94, 94)
CD:
  Matrix shape: (94, 94)
  Significant edges: 100

DEBUG UC: shape = (93, 93)
UC:
  Matrix shape: (93, 93)
  Significant edges: 101

DEBUG Healthy: shape = (101, 101)
Healthy:
  Matrix shape: (101, 101)
  Significant edges: 223


In [30]:
import os
import shutil

config = {
    "CD": ("sparcc_CD", "CD"),
    "UC": ("sparcc_UC", "UC"),
    "Healthy": ("sparcc_Healthy", "Healthy")
}

os.makedirs("/content/sparcc_results", exist_ok=True)

for cond, (folder, fname) in config.items():
    base = f"/content/fastspar/{folder}"

    cor_file = f"{base}/cor_{fname}.csv"
    pval_file = f"{base}/pvals_{fname}.csv"

    if os.path.exists(cor_file):
        shutil.copy(cor_file, "/content/sparcc_results/")
        print(f"cor_{fname}.csv")
    else:
        print(f"Missing: {cor_file}")

    if os.path.exists(pval_file):
        shutil.copy(pval_file, "/content/sparcc_results/")
        print(f"pvals_{fname}.csv")
    else:
        print(f"Missing: {pval_file}")

shutil.make_archive("/content/sparcc_final", "zip", "/content/sparcc_results")


cor_CD.csv
pvals_CD.csv
cor_UC.csv
pvals_UC.csv
cor_Healthy.csv
pvals_Healthy.csv


'/content/sparcc_final.zip'

In [28]:
import os

for cond in ["sparcc_CD", "sparcc_UC", "sparcc_Healthy"]:
    print("\n", cond)
    print(os.listdir(f"/content/fastspar/{cond}"))


 sparcc_CD
['bootstrap_cors', 'bootstrap_corr_only', 'cor_CD.csv', 'bootstrap_counts', 'cov_CD.csv', 'pvals_CD.csv']

 sparcc_UC
['bootstrap_cors', 'bootstrap_corr_only', 'pvals_UC.csv', 'bootstrap_counts', 'cov_UC.csv', 'cor_UC.csv']

 sparcc_Healthy
['bootstrap_cors', 'bootstrap_corr_only', 'pvals_Healthy.csv', 'cor_Healthy.csv', 'bootstrap_counts', 'cov_Healthy.csv']
